# Dataset Split Distribution and Multi-Nodule Slice Analysis

This notebook analyzes:

- `001_holdout_split_lidc_lndb.csv` for the **train**, **validation**, and **test** splits.
- `004_classification_cv_5fold_seed42.csv` for each cross-validation fold.
- CT slices containing more than one nodule, analyzed separately for **LNDb** and **LIDC-IDRI**.

> **Note:** each CSV row represents one nodule-slice pair. Therefore, `sample_rows` may be greater than `unique_ct_slices` when a slice contains multiple nodules.

## 1. Notebook Configuration and Setup

All display and visualization settings are centralized below so repeated runs produce consistent output.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

NOTEBOOK_CONFIG = {
    "random_seed": 42,
    "figure_dpi": 120,
    "savefig_dpi": 300,
    "font_size": 10,
    "display_max_columns": None,
    "display_max_colwidth": 120,
    "dataset_order": ["LIDC-IDRI", "LNDb"],
    "label_order": ["Benign", "Malignant"],
    "cv_partition_order": ["train", "validation", "test"],
    "label_colors": {"Benign": "#4C78A8", "Malignant": "#E45756"},
    "multi_bar_colors": ["#72B7B2", "#F2CF5B", "#E45756", "#B279A2"],
    "mask_overlay_colors": ["#FF3B30", "#00C7BE", "#FFD60A", "#BF5AF2", "#FF9F0A"],
    "holdout_figure_size": (8, 4.5),
    "cv_figure_size": (11, 4.8),
    "multi_study_figure_width": 10,
    "multi_study_row_height": 0.35,
    "multi_study_top_n": 20,
    "gallery_columns": 3,
    "gallery_panel_size": (5, 5.5),
    "gallery_legend_height_ratio": 0.16,
    "gallery_max_examples": 6,
}

np.random.seed(NOTEBOOK_CONFIG["random_seed"])
pd.set_option("display.max_columns", NOTEBOOK_CONFIG["display_max_columns"])
pd.set_option("display.max_colwidth", NOTEBOOK_CONFIG["display_max_colwidth"])
plt.rcParams.update({
    "figure.dpi": NOTEBOOK_CONFIG["figure_dpi"],
    "savefig.dpi": NOTEBOOK_CONFIG["savefig_dpi"],
    "font.size": NOTEBOOK_CONFIG["font_size"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "000_dataset" / "_segmentation_dataset_v2").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")

PROJECT_ROOT = find_project_root()
DATASET_DIR = PROJECT_ROOT / "000_dataset" / "_segmentation_dataset_v2"
HOLDOUT_CSV = DATASET_DIR / "001_holdout_split_lidc_lndb.csv"
CV_CSV = DATASET_DIR / "004_classification_cv_5fold_seed42.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")

In [ ]:
holdout_df = pd.read_csv(HOLDOUT_CSV)
cv_df = pd.read_csv(CV_CSV)

required_holdout = {"dataset", "patient_id", "filename", "label", "split", "mask_path", "ct_windowed_path"}
required_cv = required_holdout | {"cv_role", "cv_fold", "cv_nodule_id"}
assert not (required_holdout - set(holdout_df.columns)), f"Missing holdout columns: {required_holdout - set(holdout_df.columns)}"
assert not (required_cv - set(cv_df.columns)), f"Missing CV columns: {required_cv - set(cv_df.columns)}"
assert len(holdout_df) == len(cv_df), "The holdout and CV row counts do not match."

print(f"Holdout: {len(holdout_df):,} rows x {holdout_df.shape[1]} columns")
print(f"CV     : {len(cv_df):,} rows x {cv_df.shape[1]} columns")
display(holdout_df.head())

## 2. Prepare Study, Nodule, and Slice Identifiers

For LIDC-IDRI, `study_id` retains the patient identifier, Study Instance UID, and Series Instance UID from the filename. For LNDb, the patient identifier is used as `study_id`.

In [ ]:
FILENAME_PATTERN = re.compile(
    r"^(?P<study_id>.+)_(?P<nodule_token>(?:cluster|finding)_\d+)_slice_(?P<slice_number>\d+)\.npy$"
)

def add_identifiers(df):
    parsed = df["filename"].str.extract(FILENAME_PATTERN)
    if parsed.isna().any().any():
        invalid = df.loc[parsed.isna().any(axis=1), "filename"].head().tolist()
        raise ValueError(f"Filenames do not match the expected pattern: {invalid}")

    result = df.copy()
    result["study_id"] = parsed["study_id"]
    result["nodule_token"] = parsed["nodule_token"]
    result["nodule_id_parsed"] = parsed["study_id"] + "_" + parsed["nodule_token"]
    result["slice_number"] = parsed["slice_number"].astype(int)
    result["ct_slice_id"] = (
        result["dataset"].astype(str) + "::"
        + result["study_id"] + "::slice_"
        + result["slice_number"].astype(str)
    )
    return result

holdout = add_identifiers(holdout_df)
cv = add_identifiers(cv_df)
display(holdout[["dataset", "patient_id", "study_id", "nodule_id_parsed", "slice_number", "split"]].head())

## 3. Distribution Table and Visualization Functions

In [ ]:
LABEL_ORDER = NOTEBOOK_CONFIG["label_order"]
DATASET_ORDER = NOTEBOOK_CONFIG["dataset_order"]
CV_PARTITION_ORDER = NOTEBOOK_CONFIG["cv_partition_order"]
LABEL_COLORS = NOTEBOOK_CONFIG["label_colors"]

def distribution_table(data):
    grouped = (
        data.groupby(["dataset", "label"], observed=True)
        .agg(
            sample_rows=("filename", "size"),
            unique_ct_slices=("ct_slice_id", "nunique"),
            nodules=("nodule_id_parsed", "nunique"),
            studies=("study_id", "nunique"),
            patients=("patient_id", "nunique"),
        )
        .reset_index()
    )
    grouped["sample_pct_within_dataset"] = (
        grouped["sample_rows"] / grouped.groupby("dataset")["sample_rows"].transform("sum") * 100
    ).round(2)
    return grouped.sort_values(["dataset", "label"]).reset_index(drop=True)

def plot_label_distribution(data, title):
    counts = (data.groupby(["dataset", "label"]).size().unstack(fill_value=0)
              .reindex(index=DATASET_ORDER, columns=LABEL_ORDER, fill_value=0))
    ax = counts.plot(
        kind="bar", stacked=True, figsize=NOTEBOOK_CONFIG["holdout_figure_size"], rot=0,
        color=[LABEL_COLORS[label] for label in counts.columns], edgecolor="white"
    )
    ax.set_title(title)
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Number of nodule-slice samples")
    ax.legend(title="Label")
    for container in ax.containers:
        labels = [f"{int(value):,}" if value > 0 else "" for value in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="center", color="white", fontsize=9)
    plt.tight_layout()
    plt.show()

def show_distribution(data, title):
    if data.empty:
        print("No data are available for this section.")
        return
    display(distribution_table(data))  # Display the table before the visualization.
    plot_label_distribution(data, title)


## 4. Holdout split — Train

In [ ]:
train_data = holdout.loc[holdout["split"].eq("train")]
show_distribution(train_data, "Label Distribution — Holdout Train")

## 5. Holdout split — Validation

In [ ]:
val_data = holdout.loc[holdout["split"].eq("val")]
show_distribution(val_data, "Label Distribution — Holdout Validation")

## 6. Holdout split — Test

In [ ]:
test_data = holdout.loc[holdout["split"].eq("test")]
show_distribution(test_data, "Label Distribution — Holdout Test")

## 7. Cross-validation 5-fold

For fold `k`, development rows where `cv_fold != k` form the training partition, rows where `cv_fold == k` form the validation partition, and rows where `cv_role == 'holdout_test'` remain in the test partition. Each fold has its own section, where the table is displayed before the visualization.

In [ ]:
def make_cv_partition(data, validation_fold):
    partition = data.copy()
    partition["cv_partition"] = np.select(
        [
            partition["cv_role"].eq("holdout_test"),
            partition["cv_fold"].eq(validation_fold),
        ],
        ["test", "validation"],
        default="train",
    )
    return partition

def cv_distribution_table(partition):
    return (
        partition.groupby(["cv_partition", "dataset", "label"], observed=True)
        .agg(
            sample_rows=("filename", "size"),
            unique_ct_slices=("ct_slice_id", "nunique"),
            nodules=("nodule_id_parsed", "nunique"),
            patients=("patient_id", "nunique"),
        )
        .reset_index()
        .sort_values(["cv_partition", "dataset", "label"])
        .reset_index(drop=True)
    )

def plot_cv_distribution(partition, fold):
    counts = (partition.groupby(["cv_partition", "dataset", "label"]).size()
              .unstack(fill_value=0).reindex(columns=LABEL_ORDER, fill_value=0))
    expected_index = pd.MultiIndex.from_product(
        [CV_PARTITION_ORDER, DATASET_ORDER], names=["cv_partition", "dataset"]
    )
    counts = counts.reindex(expected_index, fill_value=0)
    counts.index = [f"{role}\n{dataset}" for role, dataset in counts.index]
    ax = counts.plot(kind="bar", stacked=True, figsize=NOTEBOOK_CONFIG["cv_figure_size"], rot=0,
                     color=[LABEL_COLORS[label] for label in counts.columns], edgecolor="white")
    ax.set_title(f"Label Distribution — CV Fold {fold}")
    ax.set_xlabel("Partition / Dataset")
    ax.set_ylabel("Number of nodule-slice samples")
    ax.legend(title="Label")
    for container in ax.containers:
        labels = [f"{int(value):,}" if value > 0 else "" for value in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="center", color="white", fontsize=8)
    plt.tight_layout()
    plt.show()

def show_cv_fold(fold):
    partition = make_cv_partition(cv, fold)
    display(cv_distribution_table(partition))
    plot_cv_distribution(partition, fold)
    return partition


### 7.1 CV Fold 0

In [ ]:
cv_fold_0 = show_cv_fold(0)

### 7.2 CV Fold 1

In [ ]:
cv_fold_1 = show_cv_fold(1)

### 7.3 CV Fold 2

In [ ]:
cv_fold_2 = show_cv_fold(2)

### 7.4 CV Fold 3

In [ ]:
cv_fold_3 = show_cv_fold(3)

### 7.5 CV Fold 4

In [ ]:
cv_fold_4 = show_cv_fold(4)

## 8. Identify Slices Containing More Than One Nodule

This analysis uses the holdout CSV because the CV file contains the same samples. A slice is considered multi-nodule when a `(dataset, study_id, slice_number)` combination has more than one distinct `nodule_id_parsed`.

In [ ]:
slice_nodule_counts = (
    holdout.groupby(["dataset", "study_id", "patient_id", "slice_number", "split"], observed=True)
    .agg(
        nodule_count=("nodule_id_parsed", "nunique"),
        nodule_ids=("nodule_id_parsed", lambda values: tuple(sorted(set(values)))),
        labels=("label", lambda values: tuple(sorted(set(values)))),
    )
    .reset_index()
)
multi_nodule_slices = (slice_nodule_counts.loc[slice_nodule_counts["nodule_count"].gt(1)]
                       .sort_values(["dataset", "study_id", "slice_number"])
                       .reset_index(drop=True))

multi_summary = (
    multi_nodule_slices.groupby("dataset")
    .agg(studies=("study_id", "nunique"), multi_nodule_slices=("slice_number", "size"),
         maximum_nodules_in_one_slice=("nodule_count", "max"))
    .reindex(DATASET_ORDER)
)
display(multi_summary)
display(multi_nodule_slices.head(10))

### 8.1 Study and Slice Number List — LNDb

In [ ]:
def study_slice_list(data, dataset_name):
    subset = data.loc[data["dataset"].eq(dataset_name)]
    return (
        subset.groupby(["dataset", "study_id", "patient_id", "split"], observed=True)
        .agg(
            multi_nodule_slice_count=("slice_number", "size"),
            slice_numbers=("slice_number", lambda values: tuple(sorted(set(values)))),
            max_nodules_per_slice=("nodule_count", "max"),
        )
        .reset_index()
        .sort_values(["multi_nodule_slice_count", "study_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

lndb_study_slices = study_slice_list(multi_nodule_slices, "LNDb")
display(lndb_study_slices)

### 8.2 Study and Slice Number List — LIDC-IDRI

In [ ]:
lidc_study_slices = study_slice_list(multi_nodule_slices, "LIDC-IDRI")
display(lidc_study_slices)

### 8.3 Distribution of Multi-Nodule Slices by Study

In [ ]:
def plot_multi_nodule_studies(data, dataset_name, top_n=None):
    top_n = top_n or NOTEBOOK_CONFIG["multi_study_top_n"]
    counts = (
        data.loc[data["dataset"].eq(dataset_name)]
        .groupby(["study_id", "nodule_count"]).size().unstack(fill_value=0)
    )
    top_studies = counts.sum(axis=1).nlargest(top_n).index
    plot_data = counts.loc[top_studies]
    plot_data = plot_data.loc[plot_data.sum(axis=1).sort_values().index]
    colors = NOTEBOOK_CONFIG["multi_bar_colors"][:len(plot_data.columns)]
    ax = plot_data.plot(
        kind="barh", stacked=True,
        figsize=(NOTEBOOK_CONFIG["multi_study_figure_width"],
                 max(4, NOTEBOOK_CONFIG["multi_study_row_height"] * len(plot_data))),
        color=colors, edgecolor="white"
    )
    for container in ax.containers:
        labels = [f"{int(value)}" if value > 0 else "" for value in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="center", fontsize=8)
    ax.set_title(f"Top {min(top_n, len(plot_data))} {dataset_name} studies by number of multi-nodule slices")
    ax.set_xlabel("Number of slices containing >1 nodule")
    ax.set_ylabel("Study ID")
    ax.legend(title="Nodules per slice")
    plt.tight_layout()
    plt.show()

plot_multi_nodule_studies(multi_nodule_slices, "LNDb")
plot_multi_nodule_studies(multi_nodule_slices, "LIDC-IDRI")

## 9. Multi-Nodule CT Slice Gallery with Mask Overlays

Each panel shows one CT slice. Different contour colors represent different nodule masks. The legend is rendered in a dedicated area below each CT image so it does not obscure the scan. Change `NOTEBOOK_CONFIG['gallery_max_examples']` to display more or fewer examples.

In [ ]:
def resolve_data_path(relative_or_absolute):
    path = Path(relative_or_absolute)
    return path if path.is_absolute() else DATASET_DIR / path

def normalize_image(image):
    image = np.squeeze(np.asarray(image)).astype(float)
    low, high = np.percentile(image, [1, 99])
    if high <= low:
        return np.zeros_like(image)
    return np.clip((image - low) / (high - low), 0, 1)

def rows_for_multi_slice(dataset_name, study_id, slice_number):
    return holdout.loc[
        holdout["dataset"].eq(dataset_name)
        & holdout["study_id"].eq(study_id)
        & holdout["slice_number"].eq(slice_number)
    ].sort_values("nodule_id_parsed")

def show_multi_nodule_gallery(dataset_name, max_examples=None):
    max_examples = max_examples or NOTEBOOK_CONFIG["gallery_max_examples"]
    examples = (multi_nodule_slices.loc[multi_nodule_slices["dataset"].eq(dataset_name)]
                .drop_duplicates("study_id").head(max_examples))
    if examples.empty:
        print(f"No multi-nodule slices were found for {dataset_name}.")
        return

    ncols = min(NOTEBOOK_CONFIG["gallery_columns"], len(examples))
    nrows = int(np.ceil(len(examples) / ncols))
    panel_width, panel_height = NOTEBOOK_CONFIG["gallery_panel_size"]
    fig = plt.figure(
        figsize=(panel_width * ncols, panel_height * nrows),
        constrained_layout=True,
    )
    grid = fig.add_gridspec(nrows, ncols)
    colors = NOTEBOOK_CONFIG["mask_overlay_colors"]

    for index, example in enumerate(examples.itertuples(index=False)):
        row_index, column_index = divmod(index, ncols)
        panel = grid[row_index, column_index].subgridspec(
            2, 1, height_ratios=[1, NOTEBOOK_CONFIG["gallery_legend_height_ratio"]], hspace=0.02
        )
        ax = fig.add_subplot(panel[0])
        legend_ax = fig.add_subplot(panel[1])
        rows = rows_for_multi_slice(dataset_name, example.study_id, example.slice_number)
        ct = normalize_image(np.load(resolve_data_path(rows.iloc[0]["ct_windowed_path"])))
        ax.imshow(ct, cmap="gray")
        legend_handles = []
        for color, (_, row) in zip(colors, rows.iterrows()):
            mask = np.squeeze(np.load(resolve_data_path(row["mask_path"]))) > 0
            if mask.any():
                ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=1.5)
            handle = plt.Line2D([0], [0], color=color, lw=2, label=f'{row["nodule_token"]} ({row["label"]})')
            legend_handles.append(handle)
        ax.set_title(f"{example.study_id} | slice {example.slice_number}\n{example.nodule_count} nodules", fontsize=10)
        ax.axis("off")
        legend_ax.legend(
            handles=legend_handles, loc="center", ncol=min(3, len(legend_handles)),
            fontsize=8, frameon=False, handlelength=2.5
        )
        legend_ax.axis("off")

    for index in range(len(examples), nrows * ncols):
        row_index, column_index = divmod(index, ncols)
        empty_ax = fig.add_subplot(grid[row_index, column_index])
        empty_ax.axis("off")
    fig.suptitle(f"Multi-Nodule CT Slice Examples — {dataset_name}", fontsize=14)
    plt.show()

show_multi_nodule_gallery("LNDb")
show_multi_nodule_gallery("LIDC-IDRI")

## 10. Optional: Save the Multi-Nodule Slice Lists

The following cell is disabled by default so that running the notebook does not create additional files. Uncomment it to export the results.

In [ ]:
# OUTPUT_DIR = PROJECT_ROOT / "001_7_dataset_split"
# lndb_study_slices.to_csv(OUTPUT_DIR / "multi_nodule_slices_lndb.csv", index=False)
# lidc_study_slices.to_csv(OUTPUT_DIR / "multi_nodule_slices_lidc_idri.csv", index=False)
# multi_nodule_slices.to_csv(OUTPUT_DIR / "multi_nodule_slice_details.csv", index=False)